<a href="https://colab.research.google.com/github/em2tech/Machine_Learning_SJU/blob/main/H6W/HW6_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**HW6 — Analysis Report: PCA on the Olivetti Faces Dataset**

**1. How many principal components were required to retain 95% of the total variance compared to the original 4,096 dimensions? Based on your reconstruction plots in Task 3, what visual details are lost at lower values of k; e.g., k=10, and at what point does the reconstructed image become visually indistinguishable from the original?**

**Answer:**
Fitted on all 400 images, PCA needed 123 components to keep 95% of the total variance. That is just 3% of the original 4,096 dimensions, a 33.3 x compression. By comparison, 66 components keep 90% of the variance and 260 keep 99%. The number is small for two reasons. First, pixels are highly redundant: neighboring pixels have similar values, and all faces share the same basic layout. Second, with only 400 images, PCA can find at most about 400 useful components. Returns also diminish: going from 90% to 95% almost doubles the component count (66 → 123), and going to 99% roughly doubles it again (260), because the last bit of variance is mostly small, image-specific details and noise.
At k = 10, the reconstructions look like blurry, generic faces, with the highest error of any threshold (MSE 0.0062). Identity details, fine texture and expressions are lost. Subject 33's glasses almost disappear, while faint "ghost" glasses show up on people who don't wear any. Only overall brightness, lighting direction, head position and the rough face outline remain. Detail returns steadily as k grows: faces are recognizable at k = 50 (MSE 0.0024), close to the original at k = 100 (MSE 0.0013), nearly identical at k = 200 (MSE 0.00045), and visually indistinguishable at k = 300 (MSE 0.0001). That point is well above the 95% variance mark (123 components) and close to the 99% mark (260). So 95% of the variance is enough to recognize the faces, but keeping their fine details takes many more components.


**2. What do the earliest principal components (the first few "Eigenfaces") physically capture in the images? Why do later principal components appear increasingly like fine-grained pixel noise?**

**Answers:**   The first eigenfaces show the largest, most widespread differences across the 400 images, and these mostly reflect imaging conditions and overall shape rather than identity. PC 1 (23.81% of the variance) is almost uniform across the whole face, so it captures overall brightness and contrast. PC 2 (13.99%) splits the face into left and right halves, capturing lighting direction and small head turns. PC 3 (7.97%) contrasts the forehead and hairline with the jaw and face outline. PC 4–15 pick out specific regions: the eyes (PC 6), a clear glasses-frame pattern (PC 7), the eyebrows (PC 8 and PC 11), and the mouth and chin (PC 5, PC 13 and PC 14), which respond to expressions and beards. Because these patterns affect many neighboring pixels at once, they are smooth and cover large areas. Together, the first 15 eigenfaces explain 72.12% of all variance, and the first three alone explain 45.78%.
Later components look like fine-grained noise for three reasons. First, once the large shared patterns are taken out, what remains is small variation that belongs to only a few images (a wrinkle, a strand of hair, a reflection) or is pixel-level noise, so each component explains very little. You can already see this drop: PC 15 explains only 1.06%. Second, every component must be orthogonal to all the earlier ones, so once the smooth patterns are used up, the only directions left are increasingly checkered, high-frequency ones. Third, with only 400 images, these late components are estimated from weak signals and partly fit noise in this particular sample. This is why dropping the later components works as denoising: it removes mostly noise and image-specific detail while keeping the structure all faces share.


**3. Why is it critical to fit the PCA transformer strictly on the training split rather than on the entire dataset prior to splitting? What negative impacts occur if PCA is fitted on the full dataset before training downstream classifiers?**

**Answers:**
PCA is a learned transformation. Fitting it estimates the mean face and the principal axes, so, like any model parameters, these must be learned from the training data only. The test set is meant to stand in for brand-new images the system has never seen. In faces_pca notebook, PCA is fitted on X_train alone and then only applied to X_test, so the test images are projected onto axes they had no part in creating, just as a real face-recognition system would handle a new photo. If PCA were fitted on the full dataset first, the test images would shape the mean face and the axes. That is data leakage: it gives the classifier information about the test set that it would never have in practice and pushes the measured accuracy above what the model would really achieve. The effect is stronger with few samples and many features, as here, where each image noticeably moves the axes. Leakage also makes model comparisons and tuning unreliable, doesn't reflect how the system would be used, and breaks cross-validation unless PCA sits inside a scikit-learn Pipeline.
This run shows the leakage is real. Fitted on all 400 images, PCA needed 123 components to keep 95% of the variance. Fitted on the 320 training images alone, it needed only 111. The 80 test images change the PCA space enough to add 12 extra directions of variation. Tasks 2–4 could safely use the full dataset because they only describe and visualize the data and make no claims about prediction accuracy. From Task 5 on, where performance is measured, PCA is fitted on the training set only.


**4. Compare the classification accuracy of Logistic Regression against Kernel SVM and the Ensemble Classifier on PCA-reduced features. Did non-linear kernels or ensemble methods yield substantial accuracy gains? Explain why classical classifiers (linear, non-linear, or ensemble) hit a performance ceiling when operating directly on flattened pixel vectors or global PCA projections.**

**Answer:** No, neither the non-linear kernel nor the ensemble gave a substantial accuracy gain. All three classifiers landed within 1.25 percentage points of each other: Logistic Regression scored 96.25%, the RBF SVM 97.50%, and the ensemble 96.25%. With 80 test images, 1.25 points is exactly one face, which is within the variation you'd see from a different random split. The Task 6 results back this up. Logistic Regression and the RBF SVM disagreed on only 2 test images and failed on mostly the same hard faces. Inside the ensemble, the Random Forest was the weakest member (93.75%), because its axis-aligned splits on individual PCA coordinates suit this data poorly. Averaging it with the polynomial SVM (95.00%) pulled the ensemble below the RBF SVM alone. The ensemble's main effect was a 10.5× longer training time, mostly because calibrating the two SVMs' probabilities runs an internal 5-fold cross-validation.
The classifiers stop improving at the same point because the limit comes from the features they all share, not from the learning algorithm. With 111 features and 320 training images, the 40 subjects are already almost linearly separable. That leaves extra flexibility little to fix, and with only 8 images per person, it adds more risk of overfitting. The remaining errors come from the representation. Flattened pixels and global PCA coordinates describe where brightness falls, not what the face looks like. Flattening also throws away which pixels are neighbors, so the models can't learn local patterns like edges or eye corners. Neither representation is robust to shifts, lighting or pose: the two strongest PCA components capture brightness and lighting direction, not identity. And because PCA is unsupervised, it keeps the directions with the most variance rather than the ones that best separate people. Supervised methods such as LDA ("Fisherfaces") are designed to fix that. Since every model gets the same limited input, they all hit the same ceiling of about 96–97%.


**5.Based on your findings in Task 8 regarding accuracy and training efficiency, why are global linear transformations like PCA and classical pixel classifiers insufficient for complex computer vision tasks?**

**Answer:** Task 8 showed that PCA is a big efficiency win: Logistic Regression trained **91.7 x faster** on PCA features (0.23 s vs. 20.73 s on raw pixels) with exactly the same 96.25% accuracy. Yet all four models plateaued between **96.25% and 97.50%**, which points to the representation as the real limit. Global linear transforms and classical pixel classifiers fall short for complex vision tasks for five reasons. First, PCA is global and linear: it describes every image as a weighted sum of whole-image templates, so it can't capture the local parts (edges, textures, eyes) and non-linear combinations real images are made of. It also can't recognize the same object when it is shifted or lit differently. Second, it has no built-in robustness to position, size, rotation or lighting. Even on Olivetti, a best-case dataset of centered, aligned faces on plain backgrounds, brightness and lighting direction dominated the top two components, 37.8% of all variance. Third, flattening throws away spatial structure, so the classifier has to learn each pattern separately at every location. Fourth, PCA keeps what varies most rather than what matters for the task. Fifth, it doesn't scale: raw-pixel Logistic Regression already took 20.7 s on just 320 small grayscale images, kernel SVMs grow roughly quadratically with sample count, and calibrated ensembles multiply the cost further (10.5x here).

Modern computer vision handles these problems with learned, hierarchical, local feature extractors such as convolutional neural networks (CNNs) and vision transformers. Convolutional filters scan small neighborhoods and reuse the same weights across the whole image, which makes them robust to shifts and keeps the number of parameters low. Stacked layers build up from edges to textures to parts to whole objects through many non-linear steps. Because training is supervised, the network learns what actually separates the classes, and data augmentation and pooling make the features robust to pose and lighting. For faces, deep embedding models map the same person to nearby vectors regardless of lighting, pose or expression, which PCA can't do. In short, PCA remains valuable for compression, denoising, visualization and speeding up training, and in this assignment it cut training time by more than 90x at no cost in accuracy. But because it is global, linear, unsupervised and ignores spatial structure, it can't produce the features complex real-world vision tasks need, and better classifiers on top of the same features can't get past that limit.